```text
=============================================================================
                    ____        __
   ____ ___  __  __/ __ )__  __/ /____  _____
  / __ `__ \/ / / / __  / / / / __/ _ \/ ___/
 / / / / / / /_/ / /_/ / /_/ / /_/  __(__  )
/_/ /_/ /_/\__, /_____/\__, /\__/\___/____/
          /____/      /____/

 myBytes.com | AI, Data Science & Cloud Engineering
 Copyright (c) 2026 myBytes GmbH. All rights reserved.
 Proprietary and confidential.
=============================================================================
```

# 01_garch_baseline_exploration

**Project:** soft-commodities-forecast-benchmark
**Author:** Guido Winger
**Created:** 2026-06-09
**Status:** Research notebook — companion to the methodology note

---

# GJR-GARCH baseline on four soft commodities — exploration and diagnostics

Research notebook. It walks the methodology visibly — from data
inspection through stylized-fact checks and the ARCH-LM test to the
GJR-GARCH fit and the VaR discipline. We carry all four commodities
(cocoa, coffee, sugar, cotton) side by side.

The notebook is deliberately detailed. Readers who want to reproduce
the benchmark run the pipeline via `make reproduce` (see README);
readers who want to follow the methodology behind the backtest stay
here.

## Outline

1. Setup and data fetch
2. Descriptive statistics and stylized facts
3. ARCH-LM test — justifying the GARCH family
4. GJR-GARCH(1,1) fit with Student-t innovations
5. Walk-forward forecast and VaR backtests
6. Pre-crisis-window discipline
7. Cross-asset comparison
8. What this baseline does not show

## Licensing reminder

Yahoo Finance Terms of Service prohibit redistribution of fetched
data. This notebook downloads the data at runtime; the repository
itself ships no raw data. See [`LICENSES.md`](../LICENSES.md).

## 1 · Setup and data fetch

We fetch the four ICE Continuous Futures (cocoa, coffee, sugar,
cotton) via `yfinance` over the same window as the methodology note:
2000-01-01 through the snapshot end-date pinned in
`data_snapshot.json`. Returns are computed as log-returns and scaled
to percent — that is the convention used by the `arch` package for
numerical stability.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yfinance as yf
from arch import arch_model
from scipy import stats
from statsmodels.stats.diagnostic import het_arch

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

with (REPO_ROOT / 'data_snapshot.json').open() as f:
    SNAPSHOT = json.load(f)

ASSETS = ['cocoa', 'coffee', 'sugar', 'cotton']
TICKERS = {a: SNAPSHOT['tickers'][a]['yahoo_symbol'] for a in ASSETS}
NAMES = {a: SNAPSHOT['tickers'][a]['full_name'] for a in ASSETS}
FETCH_START = SNAPSHOT['fetch_start_date']
SNAPSHOT_END = SNAPSHOT['snapshot_end_date']

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150

print(f'Snapshot end-date: {SNAPSHOT_END}')
print(f'Tickers: {TICKERS}')

In [ ]:
def fetch_returns(ticker: str, start: str, end: str) -> pd.Series:
    """Fetch daily close prices via yfinance and compute log returns in percent."""
    raw = yf.download(ticker, start=start, end=end, progress=False, auto_adjust=False)
    if isinstance(raw.columns, pd.MultiIndex):
        close = raw['Close'][ticker]
    else:
        close = raw['Close']
    close = close.dropna()
    returns = 100.0 * np.log(close / close.shift(1))
    returns = returns.dropna()
    returns.name = 'return_pct'
    return returns

returns_by_asset = {a: fetch_returns(TICKERS[a], FETCH_START, SNAPSHOT_END) for a in ASSETS}

for a, r in returns_by_asset.items():
    print(f'{a:8s} ({TICKERS[a]}): {len(r):5d} observations, {r.index.min().date()} -> {r.index.max().date()}')

## 2 · Descriptive statistics and stylized facts

Before we can justify a GARCH model we need to check that the returns
exhibit the typical stylized facts of financial time series: weak
autocorrelation in the returns themselves, strong autocorrelation in
the squared returns (volatility clustering), heavy tails (excess
kurtosis), and often skewed distributions. We tabulate this per
commodity.

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

rows = []
for a, r in returns_by_asset.items():
    jb_stat, jb_p = stats.jarque_bera(r)
    lb_r = acorr_ljungbox(r, lags=[10], return_df=True).iloc[0]
    lb_r2 = acorr_ljungbox(r ** 2, lags=[10], return_df=True).iloc[0]
    rows.append({
        'commodity': a,
        'n_obs': len(r),
        'mean (%)': r.mean(),
        'std (%)': r.std(),
        'skew': r.skew(),
        'kurtosis_excess': r.kurtosis(),
        'JB p-value': jb_p,
        'LB(10) r p-value': lb_r['lb_pvalue'],
        'LB(10) r2 p-value': lb_r2['lb_pvalue'],
    })
stylized = pd.DataFrame(rows).set_index('commodity')
stylized.style.format({
    'mean (%)': '{:.4f}',
    'std (%)': '{:.4f}',
    'skew': '{:.3f}',
    'kurtosis_excess': '{:.2f}',
    'JB p-value': '{:.2e}',
    'LB(10) r p-value': '{:.3f}',
    'LB(10) r2 p-value': '{:.2e}',
})

Three observations from the table:

1. **Weak mean autocorrelation in the returns.** The Ljung–Box tests
   on raw returns are mostly close to p > 0.05 — the returns
   themselves are largely serially uncorrelated. This justifies the
   constant-mean specification.
2. **Volatility clustering is strong everywhere.** The Ljung–Box
   tests on the squared returns reject massively for all four
   commodities. Volatility is persistent. This is the prerequisite
   for a GARCH model.
3. **Heavy tails.** Excess kurtosis is clearly above zero
   (sometimes very clearly, for example sugar). A normality
   assumption would systematically underestimate the tails. We
   therefore use Student-t innovations.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharex='col')
for i, a in enumerate(ASSETS):
    r = returns_by_asset[a]
    axes[0, i].plot(r.index, r.values, color='steelblue', linewidth=0.5)
    axes[0, i].set_title(f'{NAMES[a]}\nReturns (%)')
    axes[0, i].set_ylabel('Return (%)')
    axes[1, i].hist(r.values, bins=80, color='steelblue', alpha=0.75, density=True)
    x = np.linspace(r.min(), r.max(), 200)
    axes[1, i].plot(x, stats.norm.pdf(x, r.mean(), r.std()), color='red', linewidth=1.0, label='Normal fit')
    axes[1, i].set_title('Empirical distribution vs. Normal fit')
    axes[1, i].set_xlabel('Return (%)')
    axes[1, i].legend(fontsize=8)
fig.suptitle('Soft-commodity returns, 2000 through snapshot — visual stylized-fact diagnostics', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

The top row shows the typical **volatility clusters**: calm
periods alternate with highly volatile episodes. The bottom row makes
the **heavy tails** visible — the empirical distribution has thicker
tails than the Normal fit (red line). For sugar the effect is
particularly pronounced; for cocoa it is more moderate.

## 3 · ARCH-LM test — justifying the GARCH family

The **ARCH-LM test** (Engle 1982) formally checks for conditional
heteroscedasticity. H0 = no ARCH effects. If p < 0.05, ARCH effects
are detected and GARCH modelling is methodologically justified.

We apply the test to the training period (2000 through end of 2018)
of each asset — the window used for the initial fit in the
methodology note.

In [ ]:
TRAIN_END = '2018-12-28'

arch_rows = []
for a in ASSETS:
    r_train = returns_by_asset[a].loc[:TRAIN_END]
    lm_stat, lm_p, f_stat, f_p = het_arch(r_train.values, nlags=10)
    arch_rows.append({
        'commodity': a,
        'n_train_obs': len(r_train),
        'LM stat': lm_stat,
        'LM p-value': lm_p,
        'F stat': f_stat,
        'F p-value': f_p,
        'ARCH detected': lm_p < 0.05,
    })
arch_table = pd.DataFrame(arch_rows).set_index('commodity')
arch_table.style.format({
    'LM stat': '{:.2f}',
    'LM p-value': '{:.2e}',
    'F stat': '{:.2f}',
    'F p-value': '{:.2e}',
})

The ARCH-LM p-values on the training period sit many orders of
magnitude below 0.05 for all four commodities — the null hypothesis
(no ARCH effects) is rejected decisively. This is the methodological
precondition that lets us apply GARCH-family models at all.

## 4 · GJR-GARCH(1,1) fit with Student-t innovations

We estimate the specification from the methodology note on the
training period. The specification is identical across all four
commodities — no per-asset tuning.

The **gamma** parameter carries the asymmetric response to negative
returns. In equity markets gamma is typically positive (leverage
effect). For soft commodities the effect is often weaker and
sometimes inverted: positive price jumps from supply shocks can
trigger a larger volatility response than negative returns. This
shows up in the estimated gamma values.

In [ ]:
fit_results = {}
for a in ASSETS:
    r_train = returns_by_asset[a].loc[:TRAIN_END]
    model = arch_model(r_train, mean='Constant', vol='GARCH', p=1, o=1, q=1, dist='studentst', rescale=False)
    res = model.fit(disp='off', cov_type='robust')
    fit_results[a] = res

params_table = pd.DataFrame({a: fit_results[a].params for a in ASSETS}).T
params_table.style.format('{:.5f}')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for i, a in enumerate(ASSETS):
    res = fit_results[a]
    cond_vol = res.conditional_volatility
    cond_vol.index = returns_by_asset[a].loc[:TRAIN_END].index[: len(cond_vol)]
    axes[0, i].plot(cond_vol.index, cond_vol.values, color='darkgreen', linewidth=0.7)
    axes[0, i].set_title(f'{NAMES[a]}\nConditional volatility (fitted)')
    axes[0, i].set_ylabel('Sigma_t (%)')
    std_resid = res.std_resid
    axes[1, i].hist(std_resid.dropna(), bins=80, color='darkgreen', alpha=0.75, density=True)
    nu = res.params.get('nu', 8.0)
    x = np.linspace(std_resid.dropna().min(), std_resid.dropna().max(), 200)
    axes[1, i].plot(x, stats.t.pdf(x, df=nu), color='red', linewidth=1.0, label=f'Student-t (nu={nu:.2f})')
    axes[1, i].plot(x, stats.norm.pdf(x), color='gray', linewidth=1.0, linestyle='--', label='Normal reference')
    axes[1, i].set_title('Standardised residuals vs. Student-t fit')
    axes[1, i].set_xlabel('Standardised residual')
    axes[1, i].legend(fontsize=8)
fig.suptitle('GJR-GARCH(1,1)-t — fitted conditional volatility and residual distribution', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

The top row shows the fitted conditional volatilities on the
training period. The cocoa spikes of 2008/09 and 2024 (at the edge of
the training period), the early-2000s coffee frost episodes and the
2010/11 sugar episode are visible.

The bottom row plots the distribution of standardised residuals
against the fitted Student-t density (red) and the Normal reference
(grey, dashed). For a well-specified model the standardised residuals
should lie closer to Student-t than to Normal — which is consistently
the case here.

## 5 · Walk-forward forecast and VaR backtests

The full backtest runs as walk-forward over the test period 2019
through snapshot end-date: for each test day the model is refitted on
all data *up to* the previous day and the 1-day VaR forecast is
evaluated against the realised return.

We report aggregate VaR coverage here. The full walk-forward pipeline
(refit every 21 days, around 280 refits per asset) runs via
`make reproduce`; in this notebook we show the conceptual mechanics
with a coarser refit grid so the notebook finishes in minutes rather
than hours.

In [ ]:
from scipy import stats as sp_stats

def walk_forward_var(returns: pd.Series, start: str, end: str, refit_every: int = 63) -> pd.DataFrame:
    """Coarse walk-forward 1-day VaR forecasting at the 95 % and 99 % levels.

    Refits every `refit_every` business days for tutorial speed; the
    production pipeline uses refit_every=21 (see configs/base.yaml).
    """
    test = returns.loc[start:end]
    rows = []
    res = None
    for i, (date, actual) in enumerate(test.items()):
        if i % refit_every == 0:
            history = returns.loc[:date].iloc[:-1]
            try:
                model = arch_model(history, mean='Constant', vol='GARCH', p=1, o=1, q=1, dist='studentst', rescale=False)
                res = model.fit(disp='off', cov_type='robust', show_warning=False)
            except Exception:
                continue
        if res is None:
            continue
        forecast = res.forecast(horizon=1, reindex=False)
        sigma_fc = float(np.sqrt(forecast.variance.values[-1, 0]))
        mu_fc = float(forecast.mean.values[-1, 0])
        nu = res.params.get('nu', 8.0)
        var_95 = mu_fc + sigma_fc * sp_stats.t.ppf(0.05, df=nu)
        var_99 = mu_fc + sigma_fc * sp_stats.t.ppf(0.01, df=nu)
        rows.append({'date': date, 'actual': actual, 'var_95': var_95, 'var_99': var_99, 'sigma': sigma_fc})
    return pd.DataFrame(rows).set_index('date')

TEST_START = '2019-01-01'
TEST_END = SNAPSHOT_END

var_forecasts = {a: walk_forward_var(returns_by_asset[a], TEST_START, TEST_END, refit_every=63) for a in ASSETS}

for a, df in var_forecasts.items():
    print(f'{a:8s}: {len(df):4d} forecasts, {df.index.min().date()} -> {df.index.max().date()}')

In [ ]:
def kupiec_pof(violations: int, n_obs: int, alpha: float) -> tuple[float, float]:
    """Kupiec's Proportion-of-Failures test."""
    if violations == 0 or violations == n_obs:
        return float('nan'), float('nan')
    pi = violations / n_obs
    lr = -2 * (
        (n_obs - violations) * np.log(1 - alpha) + violations * np.log(alpha)
        - (n_obs - violations) * np.log(1 - pi) - violations * np.log(pi)
    )
    return lr, 1 - sp_stats.chi2.cdf(lr, df=1)

rows = []
for a in ASSETS:
    df = var_forecasts[a]
    n_obs = len(df)
    v95 = (df['actual'] < df['var_95']).sum()
    v99 = (df['actual'] < df['var_99']).sum()
    lr95, p95 = kupiec_pof(v95, n_obs, 0.05)
    lr99, p99 = kupiec_pof(v99, n_obs, 0.01)
    rows.append({
        'commodity': a,
        'n_obs': n_obs,
        'violations_95': v95,
        'violation_rate_95': v95 / n_obs,
        'kupiec_p_95': p95,
        'violations_99': v99,
        'violation_rate_99': v99 / n_obs,
        'kupiec_p_99': p99,
    })
kupiec_table = pd.DataFrame(rows).set_index('commodity')
kupiec_table.style.format({
    'violation_rate_95': '{:.3%}',
    'violation_rate_99': '{:.3%}',
    'kupiec_p_95': '{:.3f}',
    'kupiec_p_99': '{:.3f}',
})

The aggregate Kupiec p-values sit above 0.05 for all four
commodities — violation frequency does not deviate significantly from
the nominal level. On the scale that the VaR-backtest discipline
covers, the baseline holds.

**Important.** That is the aggregate view. The next section makes
visible what the aggregate view hides.

## 6 · Pre-crisis-window discipline

GJR-GARCH adapts quickly after a crisis, so aggregate VaR coverage
can look good. But: did the model see the crisis *before* it
happened, or did it merely catch up afterwards?

We zoom for cocoa onto the window before the 2023/24 spike
(September 2022 through mid-2024) and plot the fitted conditional
volatility against the realised returns.

In [ ]:
PRE_CRISIS_START = '2022-09-01'
SPIKE_DATE = '2024-04-15'
ZOOM_END = '2024-06-30'

cocoa_vf = var_forecasts['cocoa']
cocoa_zoom = cocoa_vf.loc[PRE_CRISIS_START:ZOOM_END]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(cocoa_zoom.index, cocoa_zoom['actual'], color='steelblue', linewidth=0.6, label='Daily return (%)')
axes[0].plot(cocoa_zoom.index, cocoa_zoom['var_95'], color='orange', linewidth=1.2, label='1-day VaR 95 % (model)')
axes[0].plot(cocoa_zoom.index, cocoa_zoom['var_99'], color='red', linewidth=1.2, label='1-day VaR 99 % (model)')
axes[0].axvline(pd.Timestamp(SPIKE_DATE), color='black', linestyle='--', linewidth=0.8, label='2024 spike')
axes[0].set_title('Cocoa — daily return and model VaR before and around the 2023/24 spike', fontweight='bold')
axes[0].set_ylabel('Return / VaR (%)')
axes[0].legend(loc='lower left', fontsize=9)
axes[1].plot(cocoa_zoom.index, cocoa_zoom['sigma'], color='darkgreen', linewidth=1.2, label='Conditional volatility Sigma_t')
axes[1].axvline(pd.Timestamp(SPIKE_DATE), color='black', linestyle='--', linewidth=0.8)
axes[1].set_title('Conditional volatility — stays at historical levels before the spike, rises only on the spike day')
axes[1].set_ylabel('Sigma_t (%)')
axes[1].set_xlabel('Date')
axes[1].legend(loc='upper left', fontsize=9)
fig.tight_layout()
plt.show()

The second plot makes the core finding visible:

> Conditional volatility stays at historical levels before the spike
> day. It rises only **after** the spike has set in. The model issued
> no early warning.

This is not a particular weakness of our fit but the expected finding
for a classical GARCH model without regime detection. Early warning
requires a second layer.

## 7 · Cross-asset comparison

Stylized facts, persistence and VaR coverage vary noticeably across
the four commodities. We condense the comparison into a table.

The most important observation: **persistence is high everywhere
(alpha + 0.5·gamma + beta close to 1)**, meaning volatility shocks
fade slowly. This is the structural property that GJR-GARCH captures
well — and the reason aggregate VaR coverage looks good.

In [ ]:
def persistence(params: pd.Series) -> float:
    a = params.get('alpha[1]', 0)
    g = params.get('gamma[1]', 0)
    b = params.get('beta[1]', 0)
    return a + 0.5 * g + b

cross_rows = []
for a in ASSETS:
    p = fit_results[a].params
    cross_rows.append({
        'commodity': a,
        'mu': p.get('mu', float('nan')),
        'omega': p.get('omega', float('nan')),
        'alpha[1]': p.get('alpha[1]', float('nan')),
        'gamma[1]': p.get('gamma[1]', float('nan')),
        'beta[1]': p.get('beta[1]', float('nan')),
        'nu': p.get('nu', float('nan')),
        'persistence': persistence(p),
        'kupiec p (95 %)': kupiec_table.loc[a, 'kupiec_p_95'],
        'kupiec p (99 %)': kupiec_table.loc[a, 'kupiec_p_99'],
    })
cross = pd.DataFrame(cross_rows).set_index('commodity')
cross.style.format('{:.4f}')

## 8 · What this baseline does not show

Three methodological caveats, in more detail than in the companion
article:

1. **One stress episode per commodity.** The pre-crisis-window
   evaluation shows the 2023/24 spike for cocoa. For coffee, sugar
   and cotton the corresponding windows are defined in the code, but
   any lead-time statement rests on a single episode per commodity.
   Validation against further episodes is pending.
2. **Look-ahead in the event definition.** We know now which
   episodes were stress. A model that would have warned in real time
   would not have known the date in advance. The lead-time statement
   is conditional on knowing the crisis occurred — it is not a
   real-time out-of-sample detection.
3. **The specification is fixed.** We use the same specification for
   all four commodities. A per-asset specification search could yield
   marginally better fits but is a known data-snooping source and is
   deliberately avoided.

What the baseline structurally **cannot** do:

- **Regime detection.** Classical GARCH carries no latent states. An
  HMM layer or Markov switching would be needed for that.
- **Exogenous drivers.** Weather, COT reports, macroeconomic
  indicators are not considered. GARCH-MIDAS would integrate them.
- **Tail-specific treatment.** For very extreme tail-risk statements,
  EVT-POT on GARCH residuals would be the methodologically clean
  extension.

All of these layers are planned in the myBytes research programme
and will each get their own companion repository.

---

**Summary in one sentence.** A classical GJR-GARCH-t passes the VaR
discipline on all four soft commodities — and detects none of the
stress episodes in advance. Both statements are methodologically
correct; both motivate the next research stages.

→ Methodology note: https://mybytes.com/research/garch-soft-commodities-baseline-backtest
→ Truth-check protocol: https://mybytes.com/research/truth-check-protocol